# Validation: Confirm Self-Consistency +5pp on Qwen3.6-27B

Independent replication for `caiovicentino1/qwen36-27b-selfconsistency`.

**Claim to validate**: naive majority vote (N=4) delivers +5pp SuperGPQA over greedy on Qwen3.6-27B dense.

**Protocol**:
- Load Qwen3.6-27B
- 30 Stage B prompts with **seed 777** (different from original seed 42)
- Run: greedy + naive vote (4 traces, temp 0.7)
- NO probe (probe was null AUROC 0.509 in original — skipping)
- **Decision rule**:
  - delta >= +3pp → **VALIDATED**
  - delta [0, +3pp] → MARGINAL
  - delta < 0 → FAILED

**Budget**: ~3h on RTX 6000 (27B dense, no probe overhead).

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5' in CONFIG_MAPPING_NAMES
except Exception: ok = False
if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random
import numpy as np
import torch
from tqdm.auto import tqdm
from collections import Counter
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/validation_27b'); OUT.mkdir(exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.6-27B'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
HF_TARGET = 'caiovicentino1/qwen36-27b-selfconsistency'

N_TRACES = 4
TEMP = 0.7
MAX_NEW = 3500
N_PROMPTS = 30
SEED = 777  # different from original seed 42
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print('27B validation setup: 30 prompts, seed 777, +3pp threshold')

## Cell 2 — Load Qwen3.6-27B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Qwen3.6-27B loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Load 30 new prompts (seed 777, disjoint from original 27B eval seed=42)

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

# Exclude prompts used in original 27B eval (seed=42, first 80 for train + 20 for eval = first 100)
random.seed(42)
questions_42 = questions.copy()
random.shuffle(questions_42)
used = set(q['question'] for q in questions_42[:100])

# Build validation set with seed 777, excluding used
random.seed(SEED)
fresh = [q for q in questions if q['question'] not in used]
random.shuffle(fresh)
eval_set = fresh[:N_PROMPTS]
print(f'{len(eval_set)} fresh prompts (seed {SEED}, zero overlap with original 27B eval)')

## Cell 4 — Helpers

In [ ]:
def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [
        r'\\boxed\{([A-J])\}',
        r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?',
        r'^\s*\(?([A-J])\)?\s*\.?\s*$',
    ]:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    tail = post[-300:] if post else text[-300:]
    letters = re.findall(r'\b([A-J])\b', tail)
    return letters[-1] if letters else None

def fmt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = (f'Answer the following multiple-choice question. '
               f'Reason step by step, then put the letter in \\boxed{{}}.\n\n'
               f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=True)

def force_close(full_ids, prompt_len):
    gen = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx = tok(full, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suf = tok.decode(out[0, ctx.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suf

def gen_greedy(ids):
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    return extract_answer(force_close(out[0], ids.shape[1]))

def gen_naive(ids, n_traces=N_TRACES, temp=TEMP):
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=temp,
                             num_return_sequences=n_traces, top_p=0.95,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    answers = [extract_answer(force_close(out[i], ids.shape[1])) for i in range(n_traces)]
    filtered = [a for a in answers if a]
    return Counter(filtered).most_common(1)[0][0] if filtered else None

print('helpers ready')

## Cell 5 — Validate (30 prompts × 2 methods, ~3h)

In [ ]:
results = {'greedy': [], 'naive_vote': []}
t0 = time.time()

for qi, q in enumerate(tqdm(eval_set, desc='validate 27B')):
    p = fmt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536: continue
    gold = q['gold_letter']
    try:
        g = gen_greedy(ids); results['greedy'].append((g, g == gold))
    except Exception as e:
        print(f'greedy err: {e}'); results['greedy'].append((None, False))
    try:
        n_pred = gen_naive(ids)
        results['naive_vote'].append((n_pred, n_pred == gold))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); results['naive_vote'].append((None, False))
    except Exception as e:
        print(f'naive err: {e}'); results['naive_vote'].append((None, False))

    # Checkpoint every 3
    if (qi+1) % 3 == 0:
        g_acc = np.mean([r[1] for r in results['greedy']])
        n_acc = np.mean([r[1] for r in results['naive_vote']])
        delta = (n_acc - g_acc) * 100
        print(f'  [{qi+1}/{len(eval_set)}]  greedy={g_acc:.3f}  naive={n_acc:.3f}  Δ={delta:+.1f}pp')
        partial = {k: [(r[0], bool(r[1])) for r in v] for k, v in results.items()}
        with open(OUT / 'validation_partial.json', 'w') as f:
            json.dump(partial, f, indent=2)

g_acc = np.mean([r[1] for r in results['greedy']])
n_acc = np.mean([r[1] for r in results['naive_vote']])
delta_pp = (n_acc - g_acc) * 100

print(f'\n{"="*55}')
print(f'  VALIDATION RESULT (27B, n={len(eval_set)}, seed={SEED}, {(time.time()-t0)/60:.0f} min)')
print(f'{"="*55}')
print(f'  greedy:        {g_acc:.3f}  ({sum(r[1] for r in results["greedy"])}/{len(results["greedy"])})')
print(f'  naive vote:    {n_acc:.3f}  ({sum(r[1] for r in results["naive_vote"])}/{len(results["naive_vote"])})')
print(f'  delta:         {delta_pp:+.1f}pp')
print(f'{"="*55}')

if delta_pp >= 3:
    print(f'  ✅ CLAIM VALIDATED: +{delta_pp:.1f}pp >= +3pp threshold')
    print(f'     Original n=20 seed=42: +5.0pp')
    print(f'     Replication n={len(eval_set)} seed={SEED}: +{delta_pp:.1f}pp')
elif delta_pp >= 0:
    print(f'  ⚠️  MARGINAL: +{delta_pp:.1f}pp. Directional match but under threshold.')
else:
    print(f'  ❌ FAILED: {delta_pp:+.1f}pp.')
print(f'{"="*55}')

## Cell 6 — Upload validation result

In [ ]:
validation_summary = {
    'validation_type': 'independent_seed_replication',
    'model': MODEL_ID,
    'original_eval': {'seed': 42, 'n_prompts': 20, 'greedy': 0.550, 'naive_vote': 0.600, 'delta_pp': 5.0},
    'replication_eval': {
        'seed': SEED,
        'n_prompts': len(eval_set),
        'greedy': float(g_acc),
        'naive_vote': float(n_acc),
        'delta_pp': float(delta_pp),
    },
    'claim_validated': delta_pp >= 3,
    'verdict': 'VALIDATED' if delta_pp >= 3 else ('MARGINAL' if delta_pp >= 0 else 'FAILED'),
    'n_traces': N_TRACES,
    'temperature': TEMP,
    'max_new_tokens': MAX_NEW,
    'eval_date': time.strftime('%Y-%m-%d'),
}
with open(OUT / 'validation_n30_seed777.json', 'w') as f:
    json.dump(validation_summary, f, indent=2)

api = HfApi()
api.upload_file(path_or_fileobj=str(OUT/'validation_n30_seed777.json'),
                path_in_repo='validation_n30_seed777.json',
                repo_id=HF_TARGET, repo_type='model',
                commit_message=f'27B independent validation seed=777 n={len(eval_set)}: delta={delta_pp:+.1f}pp')
api.upload_file(path_or_fileobj=str(OUT/'validation_partial.json'),
                path_in_repo='validation_n30_seed777_details.json',
                repo_id=HF_TARGET, repo_type='model',
                commit_message='27B per-prompt validation details')
print(f'\n✅ Results uploaded: https://huggingface.co/{HF_TARGET}/blob/main/validation_n30_seed777.json')